# Backpropagation: How Neural Networks Learn

Backpropagation is the algorithm that computes gradients of the loss function with respect to weights, enabling learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Mean Squared Error loss function
def mse_loss(y_true, y_pred):
    """Mean Squared Error loss"""
    return np.mean((y_true - y_pred) ** 2)

def mse_loss_derivative(y_true, y_pred):
    """Derivative of MSE with respect to predictions"""
    return 2 * (y_pred - y_true) / len(y_true)

# Example: compute loss
y_true = np.array([[1], [0], [1]])
y_pred = np.array([[0.9], [0.2], [0.7]])

loss = mse_loss(y_true, y_pred)
print(f"MSE Loss: {loss:.4f}")

## The Chain Rule in Backpropagation

Backpropagation uses the chain rule to compute gradients:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial A} \cdot \frac{\partial A}{\partial Z} \cdot \frac{\partial Z}{\partial W}$$

Where:
- $L$ = loss
- $W$ = weights
- $A$ = activation
- $Z$ = pre-activation

In [ ]:
# Activation derivatives

def relu_derivative(x):
    """Derivative of ReLU"""
    return (x > 0).astype(float)

def sigmoid_derivative(x):
    """Derivative of sigmoid(x) where x is pre-activation"""
    sig = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return sig * (1 - sig)

# Example
z = np.array([0.5, -0.3, 2.0])
print(f"Sigmoid derivative at {z}: {sigmoid_derivative(z)}")
print(f"ReLU derivative at {z}: {relu_derivative(z)}")

## Full Backpropagation Implementation

In [ ]:
class NeuralNetworkWithBackprop:
    """Neural network with full forward and backward pass"""
    
    def __init__(self, layer_sizes, learning_rate=0.01):
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []
        
        np.random.seed(42)
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.01
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, X):
        self.activations = [X]
        self.z_values = []
        current = X
        
        for i in range(len(self.weights) - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = self.relu(z)
            self.activations.append(current)
        
        z = np.dot(self.weights[-1], current) + self.biases[-1]
        self.z_values.append(z)
        output = self.sigmoid(z)
        self.activations.append(output)
        
        return output
    
    def backward(self, y_true):
        """Backpropagation to compute gradients"""
        m = y_true.shape[1]  # batch size
        
        # Initialize gradients
        weight_grads = []
        bias_grads = []
        
        # Output layer: derivative of loss
        delta = 2 * (self.activations[-1] - y_true) / m
        
        # Backpropagate through layers
        for i in range(len(self.weights) - 1, -1, -1):
            # Gradient w.r.t. weights
            dW = np.dot(delta, self.activations[i].T)
            weight_grads.insert(0, dW)
            
            # Gradient w.r.t. bias
            db = np.sum(delta, axis=1, keepdims=True)
            bias_grads.insert(0, db)
            
            # Propagate delta to previous layer
            if i > 0:
                delta = np.dot(self.weights[i].T, delta)
                delta = delta * (self.z_values[i-1] > 0)  # ReLU derivative
        
        return weight_grads, bias_grads
    
    def update_weights(self, weight_grads, bias_grads):
        """Update weights using gradient descent"""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * weight_grads[i]
            self.biases[i] -= self.learning_rate * bias_grads[i]

print("Neural network with backpropagation implemented!")